### **Methodology (Summary)**

In this study, a supervised machine learning approach was used to model the relationship between lifestyle-related factors and studying efficiency. After preprocessing the dataset and converting all time-based features into numerical form, several regression algorithms with varying complexity levels were implemented and evaluated.

The dataset was repeatedly divided into training and test subsets to assess model generalization. Models were trained exclusively on the training data and evaluated on unseen test data using Root Mean Squared Error (RMSE) and the coefficient of determination (R²) as performance metrics. To ensure fair comparison, identical data splits and evaluation criteria were applied across all models.

Both linear and nonlinear regression models were considered, including Linear Regression, Polynomial Regression, k-Nearest Neighbors, Decision Tree Regression, and Support Vector Regression (SVR). For distance- and margin-based models, feature scaling was applied. Additionally, SVR hyperparameters were optimized to assess the impact of tuning on model performance.

To account for variability due to the small dataset size, model performance was further analyzed using repeated random train–test splits, allowing for comparison of average performance and stability across models.

**1-)** Reads CSV into a pandas DataFrame.

df.head() shows the first rows to confirm columns loaded correctly.

In [33]:
import pandas as pd
import numpy as np

# Option A (recommended in Colab):
# 1) Left sidebar -> Files -> Upload -> upload DSA210_Project_1.csv
# 2) Then use:
path = "/content/DSA210_Project_1.csv"

df = pd.read_csv(path)
print(df.shape)
df.head()


(30, 21)


,Days,Sleep Time,Waking Time,Sleep quality,Caffeine mg.,Screen Time Total,App 1 Name,App 1 Time,App 2 Name,App 2 Time,...,App 3 Time,Category 1 name,Category 1 Time,Category 2 name,Category 2 Time,Category 3 name,Category 3 time,Studying Time,Studying Efficiency,Studying Place
0,31.10.2025,23:55,09:20,8.0,200.0,14:29,TikTok,02:08,WhatsApp,01:00,...,00:51,Social,03:28,Productivity & Finance,02:11,Entertainment,00:55,03:16,7.0,in my room
1,1.11.2025,00:59,10:30,8.0,100.0,14:06,TikTok,01:59,Netflix,01:31,...,01:17,Social,03:48,Entertainment,01:33,Games,00:48,03:39,8.0,in my room
2,2.11.2025,00:23,09:45,8.0,230.0,13:35,Chrome,01:35,Netflix,01:24,...,01:10,Social,02:41,Entertainment,01:51,Productivity & Finance,01:40,04:45,9.0,ic
3,3.11.2025,01:30,07:39,6.0,250.0,15:03,Preview,02:25,Chrome,02:05,...,01:53,Productivity & Finance,04:55,Social,03:48,Entertainment,00:38,04:16,7.0,SL & room
4,4.11.2025,00:30,09:25,9.0,100.0,11:51,WhatsApp,02:16,TikTok,01:10,...,00:53,Social,04:20,Productivity & Finance,01:55,Games,00:41,01:05,5.0,in my room


**2-)**
The dataset has columns like "Studying Efficiency " with a space. Stripping avoids “column not found” errors.

In [34]:
df.columns = df.columns.str.strip()  # remove trailing spaces
df.columns.tolist()


['Days',
 'Sleep Time',
 'Waking Time',
 'Sleep quality',
 'Caffeine mg.',
 'Screen Time Total',
 'App 1 Name',
 'App 1 Time',
 'App 2 Name',
 'App 2 Time',
 'App 3 Name',
 'App 3 Time',
 'Category 1 name',
 'Category 1 Time',
 'Category 2 name',
 'Category 2 Time',
 'Category 3 name',
 'Category 3 time',
 'Studying Time',
 'Studying Efficiency',
 'Studying Place']

**3-)**
Creates new numeric columns ending with _min.

This is necessary because algorithms like linear regression / k-NN / trees operate on numeric features

In [35]:
def hhmm_to_minutes(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip()
    if ":" not in x:
        return np.nan
    h, m = x.split(":")
    return int(h) * 60 + int(m)

time_cols = [
    "Screen Time Total",
    "App 1 Time", "App 2 Time", "App 3 Time",
    "Category 1 Time", "Category 2 Time", "Category 3 time",
    "Studying Time"
]

for c in time_cols:
    if c in df.columns:
        df[c + "_min"] = df[c].apply(hhmm_to_minutes)

df[[c for c in df.columns if c.endswith("_min")]].head()


,Screen Time Total_min,App 1 Time_min,App 2 Time_min,App 3 Time_min,Category 1 Time_min,Category 2 Time_min,Category 3 time_min,Studying Time_min
0,869.0,128.0,60.0,51.0,208.0,131.0,55.0,196.0
1,846.0,119.0,91.0,77.0,228.0,93.0,48.0,219.0
2,815.0,95.0,84.0,70.0,161.0,111.0,100.0,285.0
3,903.0,145.0,125.0,113.0,295.0,228.0,38.0,256.0
4,711.0,136.0,70.0,53.0,260.0,115.0,41.0,65.0


**4-)** Pick target (y) and features (X)

We’ll do Regression first: predict Studying Efficiency.

In [36]:
target = "Studying Efficiency"

# Convert target to numeric (in case it reads as string)
df[target] = pd.to_numeric(df[target], errors="coerce")

# Choose a simple, sensible feature set first (you can expand later)
feature_cols = [
    "Sleep quality",
    "Caffeine mg.",
    "Screen Time Total_min",
    "Studying Time_min",
]

# Keep only columns that exist and are numeric
X = df[feature_cols].copy()
y = df[target].copy()

# Drop rows with missing values in X or y
data = pd.concat([X, y], axis=1).dropna()
X = data[feature_cols]
y = data[target]

X.describe(), y.describe()


(       Sleep quality  Caffeine mg.  Screen Time Total_min  Studying Time_min
 count      25.000000     25.000000              25.000000          25.000000
 mean        7.800000    231.160000             822.640000         294.520000
 std         1.443376    114.195987             209.859453         131.042016
 min         6.000000     -1.000000             360.000000          60.000000
 25%         6.000000    175.000000             711.000000         228.000000
 50%         8.000000    230.000000             835.000000         285.000000
 75%         9.000000    275.000000             912.000000         371.000000
 max        10.000000    475.000000            1254.000000         570.000000,
 count    25.000000
 mean      7.640000
 std       1.551344
 min       5.000000
 25%       7.000000
 50%       8.000000
 75%       9.000000
 max      10.000000
 Name: Studying Efficiency, dtype: float64)

**5-)**
Train on one part, test on unseen data to check generalization.

This is exactly the “Train/Test/Validation split” concept from the slides.

In [37]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

X_train.shape, X_test.shape


((18, 4), (7, 4))

**6)** Baseline model (always predict the mean)
Gives you a reference RMSE: real models should beat this.

RMSE is shown in regression slides as an error metric

In [39]:
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

baseline = DummyRegressor(strategy="mean")
baseline.fit(X_train, y_train)

pred_base = baseline.predict(X_test)

mse_base = mean_squared_error(y_test, pred_base)   # no squared argument
rmse_base = np.sqrt(mse_base)

rmse_base


np.float64(1.832371079844484)

**7)** Model 1 — Multiple Linear Regression

In [41]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

linreg = LinearRegression()
linreg.fit(X_train, y_train)

pred_lr = linreg.predict(X_test)

mse_lr = mean_squared_error(y_test, pred_lr)   # no squared=
rmse_lr = np.sqrt(mse_lr)

r2_lr = r2_score(y_test, pred_lr)

rmse_lr, r2_lr


(np.float64(1.0942038388625241), 0.642275487755532)

**8)** Polynomial Regression (degree 2)

In [42]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

poly2 = Pipeline([
    ("poly", PolynomialFeatures(degree=2, include_bias=False)),
    ("lr", LinearRegression())
])

poly2.fit(X_train, y_train)
pred_poly = poly2.predict(X_test)

mse_poly = mean_squared_error(y_test, pred_poly)
rmse_poly = np.sqrt(mse_poly)
r2_poly = r2_score(y_test, pred_poly)

rmse_poly, r2_poly


(np.float64(5.327306858180248), -7.479449510362809)

**9)** k-NN Regression (with scaling)

In [43]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler

knn_reg = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsRegressor(n_neighbors=5))
])

knn_reg.fit(X_train, y_train)
pred_knn = knn_reg.predict(X_test)

mse_knn = mean_squared_error(y_test, pred_knn)
rmse_knn = np.sqrt(mse_knn)
r2_knn = r2_score(y_test, pred_knn)

rmse_knn, r2_knn


(np.float64(1.3669569958749357), 0.4417073170731707)

**10)** Decision Tree Regression

In [44]:
from sklearn.tree import DecisionTreeRegressor

tree_reg = DecisionTreeRegressor(
    random_state=42,
    max_depth=3
)

tree_reg.fit(X_train, y_train)
pred_tree = tree_reg.predict(X_test)

mse_tree = mean_squared_error(y_test, pred_tree)
rmse_tree = np.sqrt(mse_tree)
r2_tree = r2_score(y_test, pred_tree)

rmse_tree, r2_tree


(np.float64(0.7071067811865476), 0.850609756097561)

**Now we compare all regression models in one results table**

In [45]:
import pandas as pd

results = pd.DataFrame({
    "Model": [
        "Baseline(mean)",
        "Linear Regression",
        "Poly Regression (deg2)",
        "kNN (k=5)",
        "Decision Tree (depth=3)"
    ],
    "RMSE": [rmse_base, rmse_lr, rmse_poly, rmse_knn, rmse_tree],
    "R2":   [np.nan,    r2_lr,   r2_poly,   r2_knn,   r2_tree]
}).sort_values("RMSE")

results


,Model,RMSE,R2
4,Decision Tree (depth=3),0.707107,0.850610
1,Linear Regression,1.094204,0.642275
3,kNN (k=5),1.366957,0.441707
0,Baseline(mean),1.832371,NaN
2,Poly Regression (deg2),5.327307,-7.479450


This is not in the lecture materials but since my dataset is small I wanted to implement SVR as well

**11)** SVR (Support Vector Regression) + evaluation

In [46]:
from sklearn.svm import SVR
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

svr = Pipeline([
    ("scaler", StandardScaler()),     # SVR needs scaling
    ("svr", SVR(kernel="rbf", C=10, epsilon=0.1, gamma="scale"))
])

svr.fit(X_train, y_train)
pred_svr = svr.predict(X_test)

mse_svr = mean_squared_error(y_test, pred_svr)
rmse_svr = np.sqrt(mse_svr)
r2_svr = r2_score(y_test, pred_svr)

rmse_svr, r2_svr


(np.float64(1.1957511303217336), 0.5727974541608569)

In [52]:
svr_default = Pipeline([
    ("scaler", StandardScaler()),
    ("svr", SVR(kernel="rbf", C=10, epsilon=0.1, gamma="scale"))
])

svr_default.fit(X_train, y_train)
pred_svr_default = svr_default.predict(X_test)

rmse_svr_default = np.sqrt(mean_squared_error(y_test, pred_svr_default))
r2_svr_default = r2_score(y_test, pred_svr_default)

print("SVR default RMSE:", rmse_svr_default)
print("SVR default R2:", r2_svr_default)


SVR default RMSE: 1.1957511303217336
SVR default R2: 0.5727974541608569


Compare all regression models (including SVR)

In [47]:
import pandas as pd
import numpy as np

results = pd.DataFrame({
    "Model": [
        "Baseline(mean)",
        "Linear Regression",
        "Poly Regression (deg2)",
        "kNN (k=5)",
        "Decision Tree (depth=3)",
        "SVR (RBF)"
    ],
    "RMSE": [rmse_base, rmse_lr, rmse_poly, rmse_knn, rmse_tree, rmse_svr],
    "R2":   [np.nan,    r2_lr,   r2_poly,   r2_knn,   r2_tree,   r2_svr]
}).sort_values("RMSE")

results


,Model,RMSE,R2
4,Decision Tree (depth=3),0.707107,0.850610
1,Linear Regression,1.094204,0.642275
5,SVR (RBF),1.195751,0.572797
3,kNN (k=5),1.366957,0.441707
0,Baseline(mean),1.832371,NaN
2,Poly Regression (deg2),5.327307,-7.479450


Since my dataset is small I am also going to tune SVR to better analyze the data

In [48]:
from sklearn.model_selection import ParameterGrid

param_grid = {
    "svr__C": [1, 10, 100],
    "svr__epsilon": [0.1, 0.2, 0.5],
    "svr__gamma": ["scale", 0.1, 1]
}

best = None

for params in ParameterGrid(param_grid):
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("svr", SVR(kernel="rbf"))
    ])
    model.set_params(**params)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2 = r2_score(y_test, pred)

    if best is None or rmse < best["rmse"]:
        best = {"params": params, "rmse": rmse, "r2": r2}

best


{'params': {'svr__C': 10, 'svr__epsilon': 0.1, 'svr__gamma': 0.1},
 'rmse': np.float64(0.8797767119289414),
 'r2': 0.7687417921965093}

In [50]:
from sklearn.svm import SVR
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# Build the tuned model using the best params you found
svr_tuned = Pipeline([
    ("scaler", StandardScaler()),
    ("svr", SVR(kernel="rbf"))
])
svr_tuned.set_params(**best["params"])

# Fit on train, evaluate on test
svr_tuned.fit(X_train, y_train)
pred_svr_tuned = svr_tuned.predict(X_test)

mse_svr_tuned = mean_squared_error(y_test, pred_svr_tuned)
rmse_svr_tuned = np.sqrt(mse_svr_tuned)
r2_svr_tuned = r2_score(y_test, pred_svr_tuned)

print("Best params:", best["params"])
print("SVR tuned RMSE:", rmse_svr_tuned)
print("SVR tuned R2:", r2_svr_tuned)


Best params: {'svr__C': 10, 'svr__epsilon': 0.1, 'svr__gamma': 0.1}
SVR tuned RMSE: 0.8797767119289414
SVR tuned R2: 0.7687417921965093


Now I will compare all models including SVR tuned

In [53]:
import pandas as pd
import numpy as np

# Fill these with YOUR existing variables:
# Example names you might have (change to yours):
# rmse_lr, r2_lr
# rmse_poly, r2_poly
# rmse_knn, r2_knn
# rmse_tree, r2_tree
# rmse_base  (baseline usually has no R2)

final_results = pd.DataFrame([
    {"Model": "Baseline(mean)",        "RMSE": rmse_base,        "R2": np.nan},

    {"Model": "Linear Regression",     "RMSE": rmse_lr,          "R2": r2_lr},
    {"Model": "Poly Regression (deg2)","RMSE": rmse_poly,        "R2": r2_poly},
    {"Model": "kNN (k=5)",             "RMSE": rmse_knn,         "R2": r2_knn},
    {"Model": "Decision Tree",         "RMSE": rmse_tree,        "R2": r2_tree},

    {"Model": "SVR (default)",         "RMSE": rmse_svr_default, "R2": r2_svr_default},
    {"Model": "SVR (tuned)",           "RMSE": rmse_svr_tuned,   "R2": r2_svr_tuned},
]).sort_values("RMSE").reset_index(drop=True)

final_results


,Model,RMSE,R2
0,Decision Tree,0.707107,0.850610
1,SVR (tuned),0.879777,0.768742
2,Linear Regression,1.094204,0.642275
3,SVR (default),1.195751,0.572797
4,kNN (k=5),1.366957,0.441707
5,Baseline(mean),1.832371,NaN
6,Poly Regression (deg2),5.327307,-7.479450


This evaluates each model across many random splits and compares average RMSE:

In [55]:
import numpy as np
import pandas as pd

from sklearn.model_selection import ShuffleSplit
from sklearn.metrics import mean_squared_error

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR

# -----------------------------
# Put your tuned SVR params here:
# Example format:
# tuned_params = {"svr__C": 10, "svr__epsilon": 0.1, "svr__gamma": 0.1}
# -----------------------------
tuned_params = best["params"]  # <-- if your tuning block stored the best dict as "best"
# If your variable is named differently, replace best["params"] with your own dict.

models = {
    "Linear": LinearRegression(),
    "Poly2": Pipeline([
        ("poly", PolynomialFeatures(2, include_bias=False)),
        ("lr", LinearRegression())
    ]),
    "kNN(k=5)": Pipeline([
        ("scaler", StandardScaler()),
        ("knn", KNeighborsRegressor(5))
    ]),
    "Tree(depth=3)": DecisionTreeRegressor(max_depth=3, random_state=42),

    # SVR default (same as your earlier default settings)
    "SVR(default)": Pipeline([
        ("scaler", StandardScaler()),
        ("svr", SVR(kernel="rbf", C=10, epsilon=0.1, gamma="scale"))
    ]),

    # SVR tuned (uses your tuned parameters)
    "SVR(tuned)": Pipeline([
        ("scaler", StandardScaler()),
        ("svr", SVR(kernel="rbf"))
    ]).set_params(**tuned_params),
}

rs = ShuffleSplit(n_splits=50, test_size=0.25, random_state=42)

rows = []
for name, model in models.items():
    rmses = []
    r2s = []
    for train_idx, test_idx in rs.split(X):
        Xtr, Xte = X.iloc[train_idx], X.iloc[test_idx]
        ytr, yte = y.iloc[train_idx], y.iloc[test_idx]

        model.fit(Xtr, ytr)
        pred = model.predict(Xte)

        rmse = np.sqrt(mean_squared_error(yte, pred))
        rmses.append(rmse)

        # R2 can be unstable for very small sets; still useful to average
        ss_res = np.sum((yte - pred) ** 2)
        ss_tot = np.sum((yte - np.mean(yte)) ** 2)
        r2 = 1 - ss_res / ss_tot if ss_tot != 0 else np.nan
        r2s.append(r2)

    rows.append({
        "Model": name,
        "RMSE_mean": float(np.mean(rmses)),
        "RMSE_std": float(np.std(rmses)),
        "R2_mean": float(np.nanmean(r2s)),
        "R2_std": float(np.nanstd(r2s)),
    })

cv_results = pd.DataFrame(rows).sort_values("RMSE_mean").reset_index(drop=True)
cv_results


,Model,RMSE_mean,RMSE_std,R2_mean,R2_std
0,Linear,1.111912,0.221991,0.282467,0.495387
1,SVR(tuned),1.195857,0.275666,0.164198,0.632175
2,SVR(default),1.270873,0.256448,0.106397,0.505294
3,kNN(k=5),1.272357,0.219530,0.104063,0.570586
4,Tree(depth=3),1.323623,0.321759,-0.052368,0.772916
5,Poly2,4.385766,2.414555,-12.029108,16.889999


## **Final Conclusions and Findings**

While the Decision Tree achieved the best performance on a single train–test split, repeated evaluation over multiple random splits reveals that its performance is highly unstable. The Decision Tree exhibits high variance and tends to overfit the small dataset.

Linear Regression, although simpler, achieves the lowest average RMSE and the most stable performance across splits, making it the most reliable model overall.

Support Vector Regression benefits from hyperparameter tuning but does not consistently outperform Linear Regression. Polynomial Regression performs poorly due to severe overfitting.